In [1]:
import numpy as np
from numpy.lib.stride_tricks import sliding_window_view
import pandas as pd
import torch
import torch.nn.functional as F
from sklearn.preprocessing import scale
from sklearn.feature_selection import mutual_info_classif
from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics.pairwise import euclidean_distances
import hypernetx as hnx

"""
    Recommended Versions
    numpy==1.24.4
    pandas==1.5.3
    torch==2.2.0+cpu
    scikit-learn==1.0.2
    hypernetx==2.2.0

"""

'\n    Recommended Versions\n    numpy==1.24.4\n    pandas==1.5.3\n    torch==2.2.0+cpu\n    sklearn==1.0.2\n    hypernetx==2.2.0\n\n'

# 1. Base Functions

## (1) Time Series Transformation Function

In [2]:
def HT(signal):
    """
    Hilbert transform.

    Parameters
    ----------
    signal : array
        Univariate time series.
    """
    n_timepoints = len(signal)
    rangeN = np.arange(n_timepoints).astype(float) + 1.0
    signalH = np.zeros(n_timepoints)  
    for k in range(1, len(signal)+1):
        signalH[k-1] = np.sum(signal[0:k-1] / (k-rangeN)[0:k-1]) + np.sum(signal[k:] / (k-rangeN)[k:])
    return signalH

def series_transform(seriesX, mode="R"):
    """
    Time Series Transformation Function

    Parameters
    ----------
    seriesX : array
        Univariate time series set.
    mode : string, default="R"
        The type of time series transform performed.
    """
        
    seriesN, seriesD, seriesL = seriesX.shape[0], seriesX.shape[1], seriesX.shape[2]
    if mode=="H":  # Hilbert transform
        seriesHX = np.zeros_like(seriesX)
        for i in range(seriesN):
            for j in range(seriesD):
                seriesHX[i, j, :] = HT(scale(seriesX[i, j, :]))
                seriesHX[i, j, :] = scale(seriesHX[i, j, :])
        return seriesHX             
        
    representation_functions = (
        lambda seriesX: seriesX, 
        lambda seriesX: F.avg_pool1d(F.pad(seriesX.diff(), (2, 2), "replicate"), 5, 1), 
        lambda seriesX: seriesX.diff(n=2),
        lambda seriesX: torch.fft.rfft(seriesX).abs())

    seriesX = torch.tensor(seriesX).float()
    
    if mode=="R":  # Raw series
        function = representation_functions[0]
    if mode=="P":  # First-order differential transform
        function = representation_functions[1]
    if mode=="S":  # Second-order differential transform
        function = representation_functions[2]
    if mode=="Y":  # Fourier transform
        function = representation_functions[3]
    
    seriesX_trans = function(seriesX)
    seriesX_trans = seriesX_trans.numpy()
    return seriesX_trans

## (2) Clustering Coefficient of Hypergraph Nodes

In [3]:
def clustering_coefficient(HG, s=1):
    """
    Calculation of clustering coefficient for hypergraph nodes.

    Parameters
    ----------
    HG : object
        Hypergraph object
    s : int, default=1
        Connection degree
    """
    
    adj_matrix = HG.adjacency_matrix(s=s).toarray()
    num_nodes = adj_matrix.shape[0]
    clustering_coeff = np.zeros(num_nodes)
    for i in range(num_nodes):
        neighbors = np.where(adj_matrix[i] == 1)[0]
        if len(neighbors) < 2:
            continue
        num_triangles = np.sum(adj_matrix[neighbors][:, neighbors]) / 2
        possible_triangles = len(neighbors) * (len(neighbors) - 1) / 2
        if possible_triangles > 0:
            clustering_coeff[i] = num_triangles / possible_triangles
    return clustering_coeff

## (3) Interval Hypergraph Generation

In [4]:
def interval_hypergraph_generation(seriesX, window_size=9, percents=[0.3, 0.4, 0.5, 0.6, 0.7]):
    """
    Convert the input time series set into a set of hypergraphs under multiple thresholds.

    Parameters
    ----------
    seriesX : array
        Univariate time series set.
    window_size : int, default=9
        The length of the interval window.
    percents : list, default=[0.3, 0.4, 0.5, 0.6, 0.7]
        Hypergraph generation threshold percents.
    
    """
    
    seiresN, seriesD, seriesL = seriesX.shape[::]
    
    if seriesL < window_size: 
        raise ValueError("The length of the time series is less than the window size.")
   
    similarity_matrix = np.zeros((seriesL-window_size+1, seriesL-window_size+1)) 
    for s in range(seiresN): 
        time_series = seriesX[s, 0, :]
        nodes = sliding_window_view(time_series, window_shape=window_size)  # Interval nodes
        nodes = np.array(nodes)
        for i in range(len(nodes)):
            nodes[i][::] = np.sort(nodes[i])[::]
        similarity_matrix_one = euclidean_distances(nodes)

        similarity_matrix[::] = similarity_matrix[::] + similarity_matrix_one[::]
    similarity_matrix = similarity_matrix / seiresN  # Average distance matrix
    np.fill_diagonal(similarity_matrix, -np.inf)
    thresholds = np.quantile(similarity_matrix, percents)  # Distance threshold
    
    HGlist = []
    for k in range(len(thresholds)): 
        edges = []
        for i in range(len(similarity_matrix)): 
            index_ = np.arange(len(similarity_matrix))[similarity_matrix[i]<=thresholds[k]] 
            edges.append(np.sort(index_))
        HG = hnx.Hypergraph(edges)  # Hypergraph generation
        HGlist.append(HG) 
    
    return HGlist

## (4) Hypergraph-based Interval Selector

In [5]:
def hypergraph_interval_selection(seriesX, interval_depth=6, percents=[0.3, 0.4, 0.5, 0.6, 0.7]):
    """
    Hypergraph-based interval selector.

    Parameters
    ----------
    seriesX : array
        Univariate time series set.
    interval_depth : int, default=6
        Interval depth.
    percents : list, default=[0.3, 0.4, 0.5, 0.6, 0.7]
        Hypergraph generation threshold percents.
    """
    
    n_cases, n_channels, n_timepoints = seriesX.shape[::]
    
    intervalList = []
    exponent = min(interval_depth, int(np.log2(n_timepoints)) + 1)
    
    for i in (2 ** np.arange(exponent)):  # The interval length is determined by an exponent.
        
        if i==1:
            intervalList.append(np.arange(n_timepoints))
            continue
            
        splitK = i
        intervalN0 = splitK * 1
        window_size = int(np.floor(n_timepoints/splitK))  # Interval length

        nodes = sliding_window_view(np.arange(n_timepoints), window_shape=window_size)  # Interval nodes
        
        if len(nodes) <= intervalN0*3: 
            for ns in range(len(nodes)):
                intervalList.append(nodes[ns])
            continue

        HGlist = interval_hypergraph_generation(seriesX, window_size=window_size, percents=percents)  # Generation of interval hypergraph sets
        featureX = []
        for HG in HGlist:
            cc_ = clustering_coefficient(HG)  # Calculate the clustering coefficient
            featureX.append(cc_)
        featureX = np.mean(featureX, axis=0)  # Calculate the average clustering coefficient under different thresholds
        
        selectedNodes = [] 
        # Maximum-based interval selection
        intervalN = intervalN0 * 2  # The number of selected intervals          
        minFeatue = np.min(featureX) 
        feature0X = np.zeros_like(featureX) 
        featureNX = np.zeros_like(featureX) 
        featureNX[::] = featureX[::] 
        minCrossL = int(np.max([1, np.ceil(window_size/2)]))
        for j in range(intervalN):
            nodeMax = np.where(featureNX==np.max(featureNX))[0][-1]  # Maximum value node
            selectedNodes.append(nodeMax) 
            feature0X[::] = featureNX[::] 
            featureNX[np.max([0, nodeMax-minCrossL]):nodeMax] = np.min(feature0X[~np.isinf(feature0X)]) - 0.001  
            featureNX[nodeMax+1:np.min([nodeMax+1+minCrossL, len(featureNX)])] =  np.min(feature0X[~np.isinf(feature0X)]) - 0.001 
            featureNX[np.isinf(feature0X)] = -np.inf
            featureNX[nodeMax] = -np.inf

        # Minimum-based interval selection
        intervalN = intervalN0 * 1  # The number of selected intervals         
        maxFeatue = np.max(featureX)  
        feature0X = np.zeros_like(featureX) 
        featureNX = np.zeros_like(featureX)
        featureNX[::] = featureX[::] 
        minCrossL = int(np.max([1, np.ceil(window_size/2)])) 
        for j in range(intervalN): 
            nodeMin = np.where(featureNX==np.min(featureNX))[0][-1]  # Minimum value node
            selectedNodes.append(nodeMin) 
            feature0X[::] = featureNX[::]  
            featureNX[np.max([0, nodeMin-minCrossL]):nodeMin] =  np.max(feature0X[~np.isinf(feature0X)]) + 0.001  
            featureNX[nodeMin+1:np.min([nodeMin+1+minCrossL, len(featureNX)])] =  np.max(feature0X[~np.isinf(feature0X)]) + 0.001  
            featureNX[np.isinf(feature0X)] = np.inf  
            featureNX[nodeMin] = np.inf
            
        selectedNodes = np.unique(selectedNodes)
        for ns in range(len(selectedNodes)): 
            intervalList.append(nodes[selectedNodes[ns]])
    
    return intervalList

## (5)  Quantile Feature Extraction

In [6]:
def find_quantiles_torch(seriesX, quantile_divisor=4):
    """
    Quantile feature extraction for a univariate time series set.

    Parameters
    ----------
    seriesX : array
        Univariate time series set.
    quantile_divisor : int, default=4
        Quantile divisor.
    """

    seriesX = torch.from_numpy(seriesX).contiguous().float()

    n = seriesX.shape[-1]

    if n == 1:
        return seriesX
    else:
        num_quantiles = 1 + (n - 1) // quantile_divisor  # The number of quantiles
        if num_quantiles == 1: 
            return seriesX.quantile(torch.tensor([0.5]), dim=-1).permute(1, 2, 0) 
        else:
            quantiles = seriesX.quantile(
                torch.linspace(0, 1, num_quantiles), dim=-1
            ).permute(1, 2, 0)  
            quantiles[..., 1::2] = quantiles[..., 1::2] - seriesX.mean(-1, keepdims=True) 
            return quantiles

## (6) Quantile Feature Extraction for Intervals

In [7]:
def quant_extracting(seriesX, intervalList, quantile_divisor=4):
    """
    Quantile feature extraction for a univariate time series set according to interval list.

    Parameters
    ----------
    seriesX : array
        Univariate time series set.
    intervalList : list
        Interval list.
    quantile_divisor : int, default=4
        Quantile divisor.        
    """

    featureX = [] 
    for i in range(len(intervalList)):  # Extract quantiles for each interval
        interval_ = intervalList[i] 
        featureX_ = find_quantiles_torch(seriesX[:, :, interval_], quantile_divisor=quantile_divisor).numpy()[:, 0, :]
        featureX.append(featureX_)
    featureX = np.hstack(featureX)
    
    return featureX

## (7) Triangle Detection

In [8]:
def triangle_detection(adj_matrix):
    """
    Detect whether there are triangles in the adjacency matrix of the hypergraph.

    Parameters
    ----------
    adj_matrix : array
        Adjacency matrix      
    """
        
    num_nodes = adj_matrix.shape[0]
    for i in range(0, num_nodes):
        presetI = i 
        neighbors = np.where(adj_matrix[i] == 1)[0] 
        if len(neighbors) < 2:
            continue
        num_triangles = np.sum(adj_matrix[neighbors, :][:, neighbors]) / 2  
        if num_triangles>0: 
            return presetI, True 
    
    return presetI, False 

## (8) Adjacency Matrix Generation

In [9]:
def adjacency_matrix_generation(hyperedges, m, s=1):
    """
    Generate the node adjacency matrix from a hyperedge set.

    Parameters
    ----------
    hyperedges : list
        Hyperedge set.
    m : int
        Number of nodes.
    s : int, default=1
        Connection degree        
    """

    adj_matrix = np.zeros((m, m), dtype=np.int32) 
    for edge in hyperedges: 
        i, j = np.meshgrid(edge, edge, indexing='ij') 
        adj_matrix[i, j] += 1
    np.fill_diagonal(adj_matrix, 0) 
    adj_matrix = adj_matrix>=s 
    
    return adj_matrix

## (9) Triangle Detection-based Feature Filter

In [10]:
def triangle_detection_feature_filter(trainX, featureScores, neighbor=3, max_iter=None, s=1):
    """
    Triangle detection-based feature filter.

    Parameters
    ----------
    trainX : array
        Feature vector training set.
    featureScores : list
        Feature scores.
    neighbor : int, default=13
        Hyperedge size.
    max_iter : int, default=None
        Maximum number of iterations.
    s : int, default=1
        Connection degree.       
    """
    
    sortIndexes = np.argsort(featureScores)
    trainX = trainX[:, sortIndexes]
    featureScores = featureScores[sortIndexes]
    
    sampleN, featureN = trainX.shape[::]
    if max_iter is None:
        max_iter = featureN
    
    corrM = np.abs(np.corrcoef(trainX, rowvar=False).astype(np.float32)) 
    corrM[np.isnan(corrM)] = 0  
    corrM[np.isinf(corrM)] = 0 
    np.fill_diagonal(corrM, np.inf)  
    
    deleteIndexes = [] 
    objList = [] 
    edges = [] 
    for i in range(featureN):
        sortIndexes_ = np.argsort(corrM[i])[::-1]  
        nodes_ = sortIndexes_[:neighbor] 
        edges.append(nodes_.astype(np.int32)) 
    
    adjM = adjacency_matrix_generation(edges, featureN, s=s)  # Generate adjacency matrix
    
    presetI, triaFlag = triangle_detection(adjM)
    
    if not triaFlag: 
        return sortIndexes[::-1][:len(adjM)]
    
    for i in range(max_iter):
        adjM = adjM[presetI+1:, :][:, presetI+1:]
        presetI, triaFlag_test = triangle_detection(adjM)  # Detect whether a triangle exists
 
        if not triaFlag_test:
            break

    outIndexes = sortIndexes[::-1][:len(adjM)]
    adjM = 0
    return outIndexes

# 2. MINE Class Function

In [11]:
""" MINEClassifier
    A novel interval-based TSC algorithm with multiview interval hypergraph learning
    Multiview Interval Node Ensemble (MINE).

"""

__maintainer__ = ["Changchun He"]
__all__ = ["MINEClassifier"]

class MINEClassifier():
    """
    Multiview Interval Node Ensemble (MINE) for Time Series Classification.

    Parameters
    ----------
    percents : list, default=[0.3, 0.4, 0.5, 0.6, 0.7]
        Hypergraph generation threshold percents of the interval selector
    neighbors : list, default=[3, 4, 5, 6, 7, 8, 9, 10]
        Hypergraph hyperedge sizes of the feature selector
    n_trees : int, default=200
        The number of trees in ExtraTrees 
    random_state: int, default=None
        Controls randomness
    
     
    References
    ----------
    .. [1] Changchun He, Xin Huo, Baohan Mi, and Songlin Chen. "Multiview Interval Hypergraph Learning
       for Time Series Classification and Its Application to Turntable Fault Diagnosis"
    """
    
    def __init__(
        self,
        percents=[0.3, 0.4, 0.5, 0.6, 0.7],
        neighbors=[3, 4, 5, 6, 7, 8, 9, 10],
        n_trees=200,
        random_state=None
    ):
        self.percents = percents
        self.neighbors = neighbors
        self.n_trees = n_trees
        self.random_state=random_state
        
    def fit(self, trainSeriesX, trainY):
        """Fit a pipeline on cases (trainSeriesX, trainY), where trainY is the target variable.

        Parameters
        ----------
        trainSeriesX : 3D np.ndarray of shape = [n_cases, n_channels, n_timepoints]
            The training data.
        trainY : array-like, shape = [n_cases]
            The class labels. Each type of label is int.

        Returns
        -------
        self :
            Reference to self.
        """
        
        # Parameter calculation
        n_timepoints = trainSeriesX.shape[2]
        quantile_divisor = int(np.ceil(np.log2(n_timepoints/8)) + 1) 
        self.quantile_divisor = np.max([quantile_divisor, 1])  # Quantile divisor
        interval_depth = int(16-np.ceil(np.log2(n_timepoints)))
        self.interval_depth = np.min([interval_depth, 8])  # Interval depth

        # Sereis transformation
        trainSeriesRX = series_transform(trainSeriesX, mode="R")  # Raw series 
        trainSeriesRFX = series_transform(trainSeriesX, mode="P")  # First-order differential transform
        trainSeriesRSX = series_transform(trainSeriesX, mode="S")  # Second-order differential transform
        trainSeriesHX = series_transform(trainSeriesX, mode="H")  # Hilbert transform
        trainSeriesHFX = series_transform(trainSeriesHX, mode="P")  # First-order differential transform after Hilbert transform
        trainSeriesHSX = series_transform(trainSeriesHX, mode="S")  # Second-order differential transform after Hilbert transform
        trainSeriesHYX = series_transform(trainSeriesHX, mode="Y")  # Fourier transform after Hilbert transform
        trainSeriesYX = series_transform(trainSeriesX, mode="Y")  # Fourier transform
        trainSeriesYFX = series_transform(trainSeriesYX, mode="P")  # First-order differential transform after Fourier transform
        trainSeriesYSX = series_transform(trainSeriesYX, mode="S")  # Second-order differential transform after Fourier transform

        # Hypergraph-based interval selection
        self.interListR  = hypergraph_interval_selection(trainSeriesRX,  interval_depth=self.interval_depth, percents=self.percents)  # 得到每个区间的开始和结束点 【改进的采用超图】
        self.interListRF = hypergraph_interval_selection(trainSeriesRFX, interval_depth=self.interval_depth, percents=self.percents)  # 得到每个区间的开始和结束点 【改进的采用超图】
        self.interListRS = hypergraph_interval_selection(trainSeriesRSX, interval_depth=self.interval_depth, percents=self.percents)  # 得到每个区间的开始和结束点 【改进的采用超图】
        self.interListH  = hypergraph_interval_selection(trainSeriesHX,  interval_depth=self.interval_depth, percents=self.percents)  # 得到每个区间的开始和结束点 【改进的采用超图】
        self.interListHF = hypergraph_interval_selection(trainSeriesHFX, interval_depth=self.interval_depth, percents=self.percents)  # 得到每个区间的开始和结束点 【改进的采用超图】
        self.interListHS = hypergraph_interval_selection(trainSeriesHSX, interval_depth=self.interval_depth, percents=self.percents)  # 得到每个区间的开始和结束点 【改进的采用超图】
        self.interListHY = hypergraph_interval_selection(trainSeriesHYX, interval_depth=self.interval_depth, percents=self.percents)  # 得到每个区间的开始和结束点 【改进的采用超图】
        self.interListY  = hypergraph_interval_selection(trainSeriesYX,  interval_depth=self.interval_depth, percents=self.percents)  # 得到每个区间的开始和结束点 【改进的采用超图】
        self.interListYF = hypergraph_interval_selection(trainSeriesYFX, interval_depth=self.interval_depth, percents=self.percents)  # 得到每个区间的开始和结束点 【改进的采用超图】
        self.interListYS = hypergraph_interval_selection(trainSeriesYSX, interval_depth=self.interval_depth, percents=self.percents)  # 得到每个区间的开始和结束点 【改进的采用超图】

        # Feature extraction
        trainRX  = quant_extracting(trainSeriesRX, self.interListR, quantile_divisor=self.quantile_divisor)
        trainRFX = quant_extracting(trainSeriesRFX, self.interListRF, quantile_divisor=self.quantile_divisor)
        trainRSX = quant_extracting(trainSeriesRSX, self.interListRS, quantile_divisor=self.quantile_divisor)
        trainHX  = quant_extracting(trainSeriesHX, self.interListH, quantile_divisor=self.quantile_divisor)
        trainHFX = quant_extracting(trainSeriesHFX, self.interListHF, quantile_divisor=self.quantile_divisor)
        trainHSX = quant_extracting(trainSeriesHSX, self.interListHS, quantile_divisor=self.quantile_divisor)
        trainHYX = quant_extracting(trainSeriesHYX, self.interListHY, quantile_divisor=self.quantile_divisor)
        trainYX  = quant_extracting(trainSeriesYX, self.interListY, quantile_divisor=self.quantile_divisor)
        trainYFX = quant_extracting(trainSeriesYFX, self.interListYF, quantile_divisor=self.quantile_divisor)
        trainYSX = quant_extracting(trainSeriesYSX, self.interListYS, quantile_divisor=self.quantile_divisor)

        trainX = np.hstack((trainRX, trainRFX, trainRSX, 
                            trainHX, trainHFX, trainHSX, trainHYX, 
                            trainYX, trainYFX, trainYSX))
        self.scalerMM = MinMaxScaler()
        self.scalerMM.fit(trainX)
        trainX = self.scalerMM.transform(trainX)
        
        # Feature selection
        scoreDoubMI0 = mutual_info_classif(trainX, trainY, random_state=self.random_state)  # Feature scoring
        scoreDoubMI0[np.isnan(scoreDoubMI0)] = 0   
        scoreDoubMI0[np.isinf(scoreDoubMI0)] = 0
        self.aboveZeroIndex = np.arange(len(scoreDoubMI0))[scoreDoubMI0>0]
        trainX = trainX[:, self.aboveZeroIndex]  
        scoreDoubMI = scoreDoubMI0[self.aboveZeroIndex] 
        
        saveNList = []
        for i in range(len(self.neighbors)):
            neighbor = self.neighbors[i]  # Hyperedge size
            for j in range(neighbor+1):
                s_fs = neighbor + j  # Connection degree
                saveIndexes = triangle_detection_feature_filter(trainX, scoreDoubMI, neighbor=neighbor, s=s_fs)  # Triangle detection-based feature filter
                saveNList.append(len(saveIndexes)) 
                if len(saveIndexes)==len(scoreDoubMI):
                    break
        saveNList.append(trainX.shape[1])
        self.saveNList = np.unique(saveNList)
        self.saveNList = self.saveNList[self.saveNList>=len(scoreDoubMI)*0.1]
        
        # Classification
        self.clfList = []
        self.indexesList = [] 
        for i in range(len(self.saveNList)):
            selectedIndexes_ = np.argsort(scoreDoubMI)[::-1][:self.saveNList[i]]
            clf_ = ExtraTreesClassifier(n_estimators=self.n_trees, random_state=self.random_state, max_features=0.1, criterion="entropy")
            clf_.fit(trainX[:, selectedIndexes_], trainY)
            self.clfList.append(clf_) 
            self.indexesList.append(selectedIndexes_)
            
            
    def predict(self, testSeriesX):
        """Predict class values of n instances in testSeriesX.

        Parameters
        ----------
        testSignalX : 3D np.ndarray of shape = [n_cases, n_channels, n_timepoints]
            The data to make predictions for testSignalX.

        Returns
        -------
        y : array-like, shape = [n_cases]
            Predicted class labels.
        """
        
        # Sereis transformation
        testSeriesRX = series_transform(testSeriesX, mode="R")  # Raw series 
        testSeriesRFX = series_transform(testSeriesX, mode="P")  # First-order differential transform
        testSeriesRSX = series_transform(testSeriesX, mode="S")  # Second-order differential transform
        testSeriesHX = series_transform(testSeriesX, mode="H")  # Hilbert transform
        testSeriesHFX = series_transform(testSeriesHX, mode="P")  # First-order differential transform after Hilbert transform
        testSeriesHSX = series_transform(testSeriesHX, mode="S")  # Second-order differential transform after Hilbert transform
        testSeriesHYX = series_transform(testSeriesHX, mode="Y")  # Fourier transform after Hilbert transform
        testSeriesYX = series_transform(testSeriesX, mode="Y")  # Fourier transform
        testSeriesYFX = series_transform(testSeriesYX, mode="P")  # First-order differential transform after Fourier transform
        testSeriesYSX = series_transform(testSeriesYX, mode="S")  # Second-order differential transform after Fourier transform

        # Feature extraction
        testRX  = quant_extracting(testSeriesRX, self.interListR, quantile_divisor=self.quantile_divisor)
        testRFX = quant_extracting(testSeriesRFX, self.interListRF, quantile_divisor=self.quantile_divisor)
        testRSX = quant_extracting(testSeriesRSX, self.interListRS, quantile_divisor=self.quantile_divisor)
        testHX  = quant_extracting(testSeriesHX, self.interListH, quantile_divisor=self.quantile_divisor)
        testHFX = quant_extracting(testSeriesHFX, self.interListHF, quantile_divisor=self.quantile_divisor)
        testHSX = quant_extracting(testSeriesHSX, self.interListHS, quantile_divisor=self.quantile_divisor)
        testHYX = quant_extracting(testSeriesHYX, self.interListHY, quantile_divisor=self.quantile_divisor)
        testYX  = quant_extracting(testSeriesYX, self.interListY, quantile_divisor=self.quantile_divisor)
        testYFX = quant_extracting(testSeriesYFX, self.interListYF, quantile_divisor=self.quantile_divisor)
        testYSX = quant_extracting(testSeriesYSX, self.interListYS, quantile_divisor=self.quantile_divisor)

        testX = np.hstack((testRX, testRFX, testRSX, 
                            testHX, testHFX, testHSX, testHYX, 
                            testYX, testYFX, testYSX))
        testX = self.scalerMM.transform(testX)    
        testX = testX[:, self.aboveZeroIndex]
        
        # Feature selection
        testPYList_P = [] 
        for i in range(len(self.saveNList)):
            selectedIndexes_ = self.indexesList[i]  
            clf_ = self.clfList[i]
            testPY_P = clf_.predict_proba(testX[:, selectedIndexes_])
            testPYList_P.append(testPY_P) 
        testPYList_P = np.array(testPYList_P)
        uniqueY = clf_.classes_
            
        if len(testPYList_P)>1:
            testPList = np.mean(testPYList_P, axis=0)
            testPY_E = uniqueY[np.argmax(testPList, axis=1)]
        else:  
            testPY_E = uniqueY[np.argmax(testPYList_P[i], axis=1)]         
        
        return testPY_E

# 3 Classification Example

In [ ]:
import aeon  # Recommended Versions 0.8.1
# Loading time series set
from aeon.datasets import load_italy_power_demand
trainSeriesX, trainY = load_italy_power_demand("TRAIN")
testSeriesX, testY = load_italy_power_demand("TEST")
# Note that the input and output labels of our classifier are in the format of int.
trainY, testY = trainY.astype(int), testY.astype(int)

# Classification
mine = MINEClassifier(random_state=0)
mine.fit(trainSeriesX, trainY)
testPY = mine.predict(testSeriesX)
acc = np.sum(testPY==testY) / len(testY)

In [13]:
print("Accuracy:", acc)

Accuracy: 0.9689018464528668
